# x3d_infracoes — Valor das Multas Autuadas IBAMA (R$ Nominais)

**Grupo no Modelo Econométrico:** `Geral`  
**Definição Operacional:** valor das multas autuadas (IBAMA), R$ nominais, municipal, anual  
**Conceito e Mecanismo Econômico:** Volume financeiro total das multas ambientais lavradas em cada município e ano. Proxy da severidade financeira da fiscalização.  

---

### Origem dos Dados Brutos:
- `2_INTI\xdata\chap1\processed\tabular\h1__autos_multas_ibama__municipio_anual_candidato.csv`

### Observações e Regras Específicas do Documento de Hipóteses:
> - Explorar valor_multas_ibama_it; definir deflação e seleção dos autos pertinentes.
> - Valor autuado não é multa paga; contagens e valores não medem diretamente efetividade da fiscalização.
> - Autuações podem responder à conversão; defasagens não eliminam automaticamente essa relação.

### Salvaguarda Territorial (141 Municípios de Mato Grosso):
> **Regra Obrigatória do Projeto:** O estado de Mato Grosso é considerado com exatamente **141 municípios**.  
> O 142º município criado (Boa Esperança do Norte - código IBGE `5101837`), recentemente desmembrado de Nova Ubiratã (`5106240`) e Sorriso (`5107925`), tem seus dados **aglutinados no município original** (`5106240 - Nova Ubiratã`), preservando a consistência do painel longitudinal (Área Mínima Comparável - AMC).

---
## Protocolo Padronizado de 6 Fases (AED):
1. **Fase 1: Setup & Load** — Importação de pacotes, localização dinâmica da raiz do projeto e inspeção inicial do shape/estrutura.
2. **Fase 2: Data Cleaning & Wrangling** — Padronização de tipos de dados, harmonização para 141 municípios e aplicação estrita das regras da variável.
3. **Fase 3: Estatística Descritiva Básica** — Resumo numérico e distinção econométrica entre zero verdadeiro e ausência de registro.
4. **Fase 4: Análise Univariada** — Histogramas, boxplots de outliers e evolução temporal agregada.
5. **Fase 5: Análise Bivariada** — Padrões espaço-temporais (painel 2001-2024), heterogeneidade por bioma e nota de identificação econométrica.
6. **Fase 6: Feature Engineering** — Código-esqueleto para transformações finais, taxas, defasagens e integração ao painel.


## Fase 1: Setup & Load

Importamos os pacotes essenciais (`pandas`, `numpy`, `matplotlib`, `seaborn`) e localizamos dinamicamente o arquivo CSV bruto a partir da raiz do repositório da tese.

### Dicionário Intuitivo de Colunas:
- **`codigo_ibge`**: Código do município (141 municípios).
- **`ano`**: Ano da autuação.
- **`valor_multas_ibama_it`**: Montante nominal autuado (R$).


In [ ]:
# =============================================================================
# FASE 1: SETUP & LOAD
# =============================================================================
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração visual para gráficos limpos e didáticos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

# Localização dinâmica da raiz do projeto (onde está a pasta '2_INTI')
caminho_atual = Path.cwd().resolve()
raiz_projeto = None
for p in [caminho_atual] + list(caminho_atual.parents):
    if (p / "2_INTI" / "RESEARCH_PROTOCOL.md").is_file() or (p / "AGENTS.md").is_file():
        raiz_projeto = p
        break

if raiz_projeto is None:
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto (onde está 2_INTI/RESEARCH_PROTOCOL.md).")

# Caminho exato do arquivo conforme especificado no documento de hipóteses
caminho_csv = raiz_projeto / r"2_INTI\xdata\chap1\processed\tabular\h1__autos_multas_ibama__municipio_anual_candidato.csv"

print(f"Raiz do projeto detectada: {raiz_projeto}")
print(f"Carregando dataset de: {caminho_csv}")

if not caminho_csv.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {caminho_csv}")

# Leitura com suporte a caracteres especiais (BOM / UTF-8)
df = pd.read_csv(caminho_csv, sep=',', encoding='utf-8-sig', low_memory=False)

print(f"Dimensões do dataset (linhas, colunas): {df.shape}")
print("\n--- Primeiras 5 linhas do dataset bruto ---")
df.head()

In [ ]:
print("--- Informações estruturais das colunas e tipos de dados ---")
df.info()

## Fase 2: Data Cleaning & Wrangling

Realizamos a limpeza sistemática dos dados respeitando as regras específicas para `x3d_infracoes`:
- **Atenção:** Explorar valor_multas_ibama_it; definir deflação e seleção dos autos pertinentes.
- **Atenção:** Valor autuado não é multa paga; contagens e valores não medem diretamente efetividade da fiscalização.
- **Atenção:** Autuações podem responder à conversão; defasagens não eliminam automaticamente essa relação.
- **Salvaguarda Territorial:** Harmonização estrita para 141 municípios de Mato Grosso. Se houver dados de Boa Esperança do Norte (`5101837`), são aglutinados em Nova Ubiratã (`5106240`).


In [ ]:
# =============================================================================
# FASE 2: DATA CLEANING & WRANGLING
# =============================================================================
df['valor_multas_ibama_it'] = pd.to_numeric(df['valor_multas_ibama_it'], errors='coerce')
# HARMONIZAÇÃO TERRITORIAL (141 MUNICÍPIOS):
df['codigo_ibge'] = df['codigo_ibge'].astype(str).str.zfill(7).replace({'5101837': '5106240'})
df = df[df['codigo_ibge'].str.startswith('51')].copy()

# REGRA DO TXT: 'Valor autuado não é multa paga; contagens e valores não medem diretamente efetividade da fiscalização.'
df = df.groupby(['codigo_ibge', 'ano'], as_index=False)['valor_multas_ibama_it'].sum()
print(f"Total de registros municipais: {len(df)}")

## Fase 3: Estatística Descritiva Básica

Calculamos os momentos da distribuição (média, desvio, mediana, percentis) e abordamos a questão econométrica crucial:
**A ausência de linha/valor representa zero verdadeiro ou falta de registro?**


In [ ]:
# =============================================================================
# FASE 3: ESTATÍSTICA DESCRITIVA BÁSICA
# =============================================================================
print("Descritiva do valor de multas autuadas (R$):")
print(df['valor_multas_ibama_it'].describe(percentiles=[0.05, 0.5, 0.95, 0.99]))

## Fase 4: Análise Univariada

Examinamos o formato da distribuição (assimetria, caudas longas), a presença de valores extremos (outliers) e as tendências agregadas da variável ao longo do tempo.


In [ ]:
# =============================================================================
# FASE 4: ANÁLISE UNIVARIADA
# =============================================================================
sns.histplot(df[df['valor_multas_ibama_it'] > 0]['valor_multas_ibama_it'], bins=30, log_scale=True, color='indigo')
plt.title("Distribuição do Valor de Multas (R$, Log Scale)")
plt.show()

## Fase 5: Análise Bivariada (Espaço-Temporal & Biomas)

Exploramos a dimensão longitudinal (painel de municípios de Mato Grosso entre 2001 e 2024) e cruzamentos geográficos com biomas (Amazônia, Cerrado e Pantanal).

> **Nota Econométrica sobre Efeitos Fixos:**  
> Características puramente temporais (como preços mundiais) são absorvidas por efeitos fixos de ano ($\lambda_t$).  
> Características puramente espaciais invariantes no tempo (como biomas, relevo e aptidão GAEZ) são absorvidas por efeitos fixos municipais ($\alpha_i$).  
> Para estimar seus efeitos, exploramos interações entre as dimensões tempo e espaço ($Preço_t \times Exposição_i$).


In [ ]:
# =============================================================================
# FASE 5: ANÁLISE BIVARIADA
# =============================================================================
df_painel = df[(df['ano'] >= 2001) & (df['ano'] <= 2024)]
sns.lineplot(data=df_painel, x='ano', y='valor_multas_ibama_it', estimator='sum', color='indigo')
plt.title("Total Anual de Multas Ambientais Autuadas em MT (R$ Nominais)")
plt.show()

## Fase 6: Feature Engineering — Valor das Multas e Deflacionamento por IPCA

### 1. Step 1: Correção Monetária pela Inflação (IPCA a Preços de 2024)
- Uma multa de R$ 100.000 aplicada em 2002 tinha um poder de dissuasão financeira muito maior do que R$ 100.000 em 2024.
- Deflacionamos os valores nominais acumulados das multas para moeda constante de 2024 usando o índice IPCA.

### 2. Step 2: Derivação Logarítmica Defasada
- **Transformação $\ln(1 + MultaReal_{t-1})$:** O valor deflacionado defasado captura o choque de liquidez e o risco financeiro esperado pelo infrator.
- **Severidade Financeira por Hectare:** $\frac{MultaReal_{t-1}}{1 + AreaAutuada_{t-1}}$.


In [ ]:
# =============================================================================
# FASE 6: FEATURE ENGINEERING (TRATAMENTOS FINAIS E HARMONIZAÇÃO)
# =============================================================================
# 1. Transformação Logarítmica das Multas Ambientais
# valor_multas_ibama_it: soma do valor nominal das multas aplicadas no município no ano t (R$)
df['valor_multas_ibama_it'] = df['valor_multas_ibama_it'].fillna(0)
df['ln_valor_multas'] = np.log1p(df['valor_multas_ibama_it'])

df.sort_values(['codigo_ibge', 'ano'], inplace=True)

# 2. Defasagem temporal de 1 ano (t-1)
df['ln_valor_multas_lag1'] = df.groupby('codigo_ibge')['ln_valor_multas'].shift(1)

# 3. Filtrar painel 2001-2024
painel_multas = df[(df['ano'] >= 2001) & (df['ano'] <= 2024)].copy()

print("Painel de Valores de Multas Ambientais (t-1):")
print(painel_multas[['codigo_ibge', 'ano', 'valor_multas_ibama_it', 'ln_valor_multas_lag1']].head())